# CardioAssist AI - Fine-tuning


## Análise e Limpeza do DataSet

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
url = "https://huggingface.co/datasets/recogna-nlp/drbode240_dataset/resolve/main/train.json"

df = pd.read_json(url)

df.head()

In [ ]:
df["question_type"].unique()

In [ ]:
amostra = (
    df.groupby("question_type", group_keys=False)
      .sample(n=10, random_state=42)[["data","question_type"]]
)

display(amostra)

In [ ]:
dados = []

for _, item in df.iterrows():

    pergunta = None
    resposta = None
    questionType = None

    for registro in item["data"]:
        if registro.get("role") == "user":
            pergunta = registro.get("content")

        elif registro.get("role") == "assistant":
            resposta = registro.get("content")

    questionType = item["question_type"]

    if pergunta and resposta:
        dados.append({
            "Tipo": "ConhecimentoSaude",
            "Pergunta": pergunta,
            "Resposta": resposta,
            "QuestionType": questionType
        })

dfSaude = pd.DataFrame(dados)

In [ ]:
display(dfSaude.head(20))

In [ ]:
dfsaudeClean = dfSaude.dropna(subset=["QuestionType"])

In [ ]:
dfsaudeClean[["Pergunta", "QuestionType"]].value_counts().loc[lambda x: x > 1]

In [ ]:
dfsaudeClean = dfsaudeClean.drop_duplicates(
    subset=["Pergunta", "QuestionType"],
    keep="first"
)

In [ ]:
dfsaudeClean["QuestionType"].value_counts()

In [ ]:
dfsaudeClean = dfsaudeClean[["Tipo", "QuestionType","Pergunta", "Resposta"]]

In [ ]:
dfsaudeClean.to_csv("dataset_cardiologia.csv", index=False, encoding="utf-8-sig")

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/FIAP/ChallengeFase3/dataset_cardiologia.csv")

In [ ]:
df.head(5)

In [ ]:
df["QuestionType"].unique()

In [ ]:
amostra = (
    df.groupby("QuestionType", group_keys=False)
      .sample(n=10, random_state=42)[["Pergunta", "Resposta","QuestionType"]]
)

display(amostra)

In [ ]:
tipo_map = {
    "information": "Responda esta pergunta que solicita informação",
    "frequency": "Responda esta pergunta de frequencia de uma doença",
    "genetic changes": "Responda esta pergunta relacionada a alterações genéticas provocadas por doenças",
    "inheritance": "Responda esta pergunta sobre a transmisão hereditária de uma doença",
    "treatment": "Responda esta pergunta que solicita informações de tratamento de uma doença",
    "symptoms": "Responda esta pergunta sobre os sintomas de uma doença",
    "causes": "Responda esta pergunta a consequência de uma deficiência ou doença",
    "exams and tests": "Responda esta pergunta sobre como realizar o diagnostico de uma doença",
}

df["Tipo"] = df["QuestionType"].map(tipo_map)

In [ ]:
df.head(10)

In [ ]:
df = df.rename(columns={"Tipo": "Instrução"})

In [ ]:
df.to_csv("/content/drive/MyDrive/FIAP/ChallengeFase3/dataset_cardiologiaV2.csv", index=False, encoding="utf-8-sig")

## Treinamento da LLM

In [ ]:
# !pip uninstall -y transformers trl unsloth
!pip install -U unsloth
!pip install transformers datasets

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import json
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

DATA_PATH = "/content/drive/MyDrive/FIAP/ChallengeFase3/dataset_cardiologiaV2.csv"
OUTPUT_PATH_DATASET = "/content/drive/MyDrive/FIAP/ChallengeFase3/dataset_cardiologiaV2.json"
max_seq_length = 1024
dtype = None
load_in_4bit = True

In [ ]:
data = pd.read_csv(DATA_PATH)

In [ ]:
def format_dataset_into_model_input(data):
    formatted_data = {
        "instruction": data["Instrução"].tolist(),
        "input": data["Pergunta"].tolist(),
        "output": data["Resposta"].tolist()
    }

    with open(OUTPUT_PATH_DATASET, "w", encoding="utf-8") as output_file:
        json.dump(formatted_data, output_file, indent=4, ensure_ascii=False)

    print(f"Dataset salvo em {OUTPUT_PATH_DATASET}")

In [ ]:
format_dataset_into_model_input(data)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
def tokenize_function(examples):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for instruction, question, answer in zip(
        examples["instruction"],
        examples["input"],
        examples["output"]
    ):

        prompt = f"""### Instrução:
                {instruction}

                ### Pergunta:
                {question}

                ### Resposta:
                """

        full_text = prompt + answer + EOS_TOKEN

        # Tokeniza somente o prompt
        prompt_tokens = tokenizer(
            prompt,
            add_special_tokens=False,
        )

        # Tokeniza prompt + resposta
        full_tokens = tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_seq_length,
        )

        input_ids = full_tokens["input_ids"]
        attention_mask = full_tokens["attention_mask"]

        prompt_length = len(prompt_tokens["input_ids"])

        # Inicialmente ignora tudo
        labels = [-100] * len(input_ids)

        # Calcula loss somente na resposta
        for i in range(prompt_length, len(input_ids)):
            labels[i] = input_ids[i]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }

In [ ]:
# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
alpaca_prompt = """### Instrução:
{}

### Pergunta:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, question, answer in zip(instructions, inputs, outputs):

        text = alpaca_prompt.format(instruction, question, answer) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset

OUTPUT_PATH_DATASET = "/content/drive/MyDrive/FIAP/ChallengeFase3/dataset_cardiologiaV2.json"

dataset = load_dataset("json", data_files=OUTPUT_PATH_DATASET, split = "train")

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
)

### Treino e teste

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=tokenized_dataset,
    args=SFTConfig(
        # dataset_text_field="text",
        max_length=max_seq_length,
        # dataset_num_proc=2,
        packing=False,

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,

        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:

FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Responda esta pergunta que solicita informação",
        "Paciente sintético com pressão 190/125 e dor torácica. Quais são os sinais de alerta?",
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True, do_sample = False)

input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

tokenizer.decode(generated_tokens,skip_special_tokens=True,).strip()

In [ ]:

FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Responda esta pergunta que solicita informação",
        "O que é ceratodermia com cabelo lanoso?", # input
        "",
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

In [ ]:
model.save_pretrained("/content/drive/MyDrive/FIAP/ChallengeFase3/lora_model") # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/FIAP/ChallengeFase3/lora_model")
